# Final Project - Kansas HMDA Home-Purchase Applications
Scott Person

## Project Overview

Among Kansas home-purchase applications that received a lending decision in 2018–2025, were denial rates higher in lower-income census tracts than in tracts at or above the area median, even when comparing applicants within similar income bands?

The unit is an application record, not a unique borrower. A denial is `action_taken == 3`; the denominator is applications with a lending decision (`action_taken` 1, 2, or 3). Withdrawn, incomplete, and other non-decision records do not belong in that denominator. Income is reported in thousands of dollars; the tract-income percentage is relative to an area median, not the applicant's income. The four applicant-income bands are under $50,000, $50,000–<$100,000, $100,000–<$150,000, and $150,000 or more.

Source: [FFIEC HMDA Data Browser](https://ffiec.cfpb.gov/data-browser/), Kansas home-purchase records for 2018–2025. See the [HMDA action-taken field definitions](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields/#action_taken). The accompanying `data/analysis_data.csv` is a seven-column extract of the eight annual files in this folder. `Download_KS_Data.ipynb` records the retrieval process. I expected below-median tracts to have higher denial rates within applicant-income bands. Equal or lower rates would have surprised me. The comparison is descriptive: it cannot tell us how an otherwise identical applicant would have been treated.

## Initial Data Review

The eight annual Kansas home-purchase files are in `data/`. In my first full-file read, `total_units` needed to be treated as text because the 2019 file mixes text and numbers there. It is not part of this analysis. The optional rebuild below reads only the seven fields I kept, so it avoids that column and leaves the committed `analysis_data.csv` untouched.

| Field | Role in this analysis |
|---|---|
| `action_taken` | Lending action. Codes 1 and 2 are non-denials with a decision; code 3 is a denial. |
| `tract_to_msa_income_percentage` | Tract income expressed as a percentage of an area median; below 100 is the lower-income tract group. |
| `income` | Reported applicant income, in thousands of dollars; used only to form bands. |
| `activity_year` | Calendar year of the reported application event. |
| `derived_msa_md` | Metro-area code; `99999` identifies the non-metropolitan segment. |
| `state_code`, `loan_purpose` | Scope checks: Kansas (`KS`) and home purchase (`1`). |

The extract has no application ID, tract ID, or underwriting fields. Matching rows may be separate applications, and these fields cannot explain a lender’s decision.

In [0]:
%pip install seaborn # Databricks does not install seaborn by default

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
def to_snake(columns):
    """Normalize source column names without changing their meaning."""
    return (pd.Index(columns).str.strip().str.lower()
            .str.replace(r'[^0-9a-z]+', '_', regex=True).str.strip('_'))

DATA_PATH = Path('data/analysis_data.csv')
source_extract = pd.read_csv(DATA_PATH)
column_map = pd.DataFrame({'source': source_extract.columns,
                           'analysis': to_snake(source_extract.columns)})
raw = source_extract.copy()
raw.columns = to_snake(raw.columns)
required = {'action_taken', 'tract_to_msa_income_percentage', 'income',
            'activity_year', 'derived_msa_md', 'state_code', 'loan_purpose'}
assert required.issubset(raw.columns), f'Missing columns: {required - set(raw.columns)}'
print('Column names before and after normalization:')
print(column_map.to_string(index=False))
print('\nFirst three application records:')
print(raw.head(3).to_string(index=False))

### Rebuilding the analysis extract (optional)

The default run reads the committed seven-column CSV. To repeat the source-to-extract step, set `REBUILD_EXTRACT` to `True`. It reads all eight annual files, prints each file's shape, selects the same seven fields, and compares the resulting row and year counts with the committed extract. It does not rewrite `analysis_data.csv`.

In [0]:
REBUILD_EXTRACT = False
if REBUILD_EXTRACT:
    annual_files = sorted(Path('data').glob('hmda_ks_*.csv'))
    assert len(annual_files) == 8, f'Expected eight annual files, found {len(annual_files)}'
    source_fields = ['action_taken', 'tract_to_msa_income_percentage', 'income',
                     'activity_year', 'derived_msa-md', 'state_code', 'loan_purpose']
    frames = []
    for file in annual_files:
        year_data = pd.read_csv(file, usecols=source_fields)
        print(f'{file.name}: {len(year_data):,} rows x {len(year_data.columns)} selected columns')
        frames.append(year_data)
    rebuilt = pd.concat([frame[source_fields] for frame in frames], ignore_index=True)
    rebuilt.columns = to_snake(rebuilt.columns)
    print('Rebuilt rows:', len(rebuilt), '| committed extract rows:', len(raw))
    print('Rebuilt year counts match extract:',
          rebuilt['activity_year'].value_counts().sort_index().equals(
              raw['activity_year'].value_counts().sort_index()))

In [0]:
def first_look(frame):
    """Shape, types, missingness, distinct values, and the key numeric fields."""
    print(f'{len(frame):,} rows x {len(frame.columns)} columns')
    summary = pd.DataFrame({
        'dtype': frame.dtypes.astype(str),
        'missing': frame.isna().sum(),
        'missing_pct': (100 * frame.isna().mean()).round(2),
        'distinct': frame.nunique(dropna=True),
    })
    print('\nColumn audit:')
    print(summary.to_string())
    print('\nIncome (in $1,000s) and tract-income percentage:')
    print(frame[['income', 'tract_to_msa_income_percentage']].describe().round(2).to_string())
    print('\nFull-row repeats after the first:', int(frame.duplicated().sum()))
    print('\nSmall-domain columns:')
    for column in frame.columns:
        if frame[column].nunique(dropna=True) <= 12:
            print(column, frame[column].value_counts(dropna=False).sort_index().to_dict())
    return summary

initial_summary = first_look(raw)

### The question before cleaning

I calculated the grouped denial counts before cleaning to make sure the selected columns could produce an answer. This is a diagnostic, not the final result: the table still includes zero tract percentages and a separate missing-income group. I restrict it to actions 1–3 so that “denial rate” has the same lending-decision denominator throughout the notebook.

In [0]:
baseline_decided = raw.loc[raw['action_taken'].isin([1, 2, 3])].copy()
labels = ['Under $50k', '$50k–<$100k', '$100k–<$150k', '$150k+']
baseline_decided['income_band'] = pd.cut(
    baseline_decided['income'], [-np.inf, 50, 100, 150, np.inf],
    right=False, labels=labels).astype('string').fillna('Missing')
baseline_decided['tract_group'] = np.where(
    baseline_decided['tract_to_msa_income_percentage'] < 100,
    'Below median', 'At or above median')
baseline_table = (baseline_decided.groupby(['income_band', 'tract_group'], observed=True)
                  ['action_taken'].agg(applications='size',
                                      denials=lambda actions: actions.eq(3).sum())
                  .reset_index())
baseline_table['denial_rate_pct'] = 100 * baseline_table['denials'] / baseline_table['applications']
print(baseline_table.round(2).to_string(index=False))

## Data Integrity Audit

### Shape, columns, and data types

Year, action, loan purpose, and metropolitan-area code are coded fields, not continuous measurements. The extract contains only Kansas home-purchase applications. Actions 1–3 form the denial-rate denominator.

In [0]:
print('Non-metropolitan code 99999:', int(raw['derived_msa_md'].eq(99999).sum()))

### Missing values

The table distinguishes actual nulls from literal text placeholders and zero. Zero is a valid value in some columns, so it is a flag for investigation, not a universal missing-value rule. An absent applicant income prevents assigning an income band; zero reported income does not. A zero tract-income percentage cannot support the intended below/above-median comparison and is excluded below.

In [0]:
audit_rows = []
for column in raw.columns:
    values = raw[column].astype('string').str.strip()
    lower = values.str.lower()
    audit_rows.append({
        'column': column,
        'null': int(raw[column].isna().sum()),
        'blank_text': int(values.eq('').sum()),
        'literal_na': int(lower.eq('na').sum()),
        'question_mark': int(values.eq('?').sum()),
        'exempt': int(lower.eq('exempt').sum()),
        'zero': int(pd.to_numeric(values, errors='coerce').eq(0).sum()),
    })
missing_audit = pd.DataFrame(audit_rows).fillna(0)
print(missing_audit.to_string(index=False))

### Duplicates

The seven retained fields produce 151,994 identical rows after the first occurrence. That is not evidence of 151,994 duplicate applications: two borrowers can share all seven values. I inspected a few matching rows below, then kept them because the extract lacks an application ID. A match on these fields cannot prove the same application was loaded twice.

In [0]:
print('Identical seven-field rows after the first:', int(raw.duplicated().sum()))
print('A few matching records (not verified duplicate applications):')
print(raw.loc[raw.duplicated(keep=False)].head(5).to_string(index=False))

### Validity checks and outliers

All records have Kansas state code, home-purchase purpose, a year from 2018 through 2025, and an action code from 1 through 8. The audit finds 8,442 zero tract-income percentages, 103 negative reported incomes, and 924 incomes above $1 million. The `99999` metro-area code identifies non-metropolitan records; I check it separately instead of treating it as an impossible number. See the [FFIEC census reference](https://www.ffiec.gov/sites/default/files/data/census/cen2024_16JLUY24.pdf) and [HMDA filing instructions](https://ffiec.cfpb.gov/documentation/fig/2025/overview).

I kept negative income values: HMDA can contain them. The maximum reported income is 2,134,000 in thousands of dollars, or $2.134 billion, which looks suspect. It still falls in the $150,000+ band whether the scale is correct or not. I did not silently cap or delete it; the result compares bands, not average income.

In [0]:
print('Unexpected actions:', int((~raw['action_taken'].isin(range(1, 9))).sum()))
print('Unexpected state:', int(raw['state_code'].ne('KS').sum()))
print('Unexpected loan purpose:', int(raw['loan_purpose'].ne(1).sum()))
print('Outside 2018–2025:', int((~raw['activity_year'].between(2018, 2025)).sum()))
print('Tract income <= 0:', int(raw['tract_to_msa_income_percentage'].le(0).sum()))
print('Missing tract income:', int(raw['tract_to_msa_income_percentage'].isna().sum()))
print('Negative incomes:', int(raw['income'].lt(0).sum()))
print('Incomes above $1m (income > 1000 in $1,000s):', int(raw['income'].gt(1000).sum()))
print('Income min/max ($1,000s):', raw['income'].min(), raw['income'].max())

### Where are the zero tract percentages?

The zero tract-income values occur in several area codes. All 872 records with a zero MSA code also have a zero tract percentage, so they leave the analysis under the same tract-percentage filter. The table shows the full breakdown rather than assuming zero is confined to one geography.

In [0]:
zero_tract = raw.loc[raw['tract_to_msa_income_percentage'].eq(0)]
print('Zero tract-income percentage by MSA code:')
print(zero_tract['derived_msa_md'].value_counts().sort_index().to_string())
msa_zero = raw.loc[raw['derived_msa_md'].eq(0)]
print('\nZero-coded MSA records:', len(msa_zero))
print('Also zero tract percentage:', int(msa_zero['tract_to_msa_income_percentage'].eq(0).sum()))
assert msa_zero['tract_to_msa_income_percentage'].eq(0).all()

## Cleaning & Remediation

The filters below make the denominator match the question. They operate on a copy and log how many applications and denials each step removes. I keep matching seven-field rows and unusual reported incomes; I exclude missing incomes rather than estimating them. I also report named metropolitan and non-metropolitan records separately.

In [0]:
def denial_rate(frame):
    return 100 * frame['action_taken'].eq(3).mean() if len(frame) else np.nan

work = raw.copy()
steps = []
def retain(label, mask):
    global work
    before = work
    work = before.loc[mask].copy()
    steps.append({
        'decision': label,
        'before': len(before),
        'removed': len(before) - len(work),
        'removed_denials': int(before['action_taken'].eq(3).sum() - work['action_taken'].eq(3).sum()),
        'remaining': len(work),
        'denial_rate_pct': denial_rate(work),
    })

retain('Decisions only: actions 1, 2, 3', work['action_taken'].isin([1, 2, 3]))
retain('Positive tract-income percentage', work['tract_to_msa_income_percentage'].gt(0))
retain('Reported applicant income', work['income'].notna())
clean = work
log = pd.DataFrame(steps)
print(log.round(2).to_string(index=False))
assert clean['action_taken'].isin([1, 2, 3]).all()
assert clean['tract_to_msa_income_percentage'].gt(0).all()
assert clean['income'].notna().all()

### Correct the source-code types

After filtering actions 1–3, I store lending action and loan purpose as categories, and MSA and state codes as strings. These are labels, not quantities to average. The derived applicant-income band created below is also an ordered pandas category. I show the before/after types to make this cleaning step visible.

In [0]:
typed_fields = ['derived_msa_md', 'state_code', 'action_taken', 'loan_purpose']
print('Before conversion:')
print(clean[typed_fields].dtypes.to_string())
clean['derived_msa_md'] = clean['derived_msa_md'].astype('string')
clean['state_code'] = clean['state_code'].astype('string')
clean['action_taken'] = clean['action_taken'].astype('category')
clean['loan_purpose'] = clean['loan_purpose'].astype('category')
print('\nAfter conversion:')
print(clean[typed_fields].dtypes.to_string())

### Cleaning decisions and their cost

- Actions outside 1–3 did not receive a lending decision. This removes 132,219 records but no denials; the rate among decided applications is 9.52%. These records are outside the question, not errors.
- Nonpositive tract-income percentages remove 3,749 decided applications, including 1,773 denials. The rate among the remaining decided applications is 9.10%. Of those 3,749 excluded decided applications, 47.29% were denied. The tract comparison therefore does not cover a group with a high denial rate.
- Missing applicant income removes another 17,910 applications, including 875 denials. The final 313,643 applications have a 9.34% denial rate. Missing income may be related to the lending decision, so the findings apply to applications with reported income.
- Repeated seven-field combinations and unusually high or low reported incomes are retained. Without an application identifier, matching rows cannot be declared duplicates; the outcome uses income bands rather than a mean, so a very high value stays in the top band without dominating a calculation.
- `derived_msa_md == 99999` is kept in the statewide result and shown separately below. Its tract-income reference may differ from a named MSA; compare named-metro results before interpreting the statewide pattern as an MSA-specific effect.

Each count is from the population left by the preceding filter.

In [0]:
labels = ['Under $50k', '$50k–<$100k', '$100k–<$150k', '$150k+']
clean['income_band'] = pd.cut(clean['income'],
    bins=[-np.inf, 50, 100, 150, np.inf], right=False, labels=labels)
clean['tract_group'] = np.where(clean['tract_to_msa_income_percentage'] < 100,
    'Below median', 'At or above median')
clean['area_type'] = np.where(clean['derived_msa_md'].eq('99999'),
    'Non-metropolitan code 99999', 'Named metropolitan area')
assert clean['income_band'].notna().all()

def rates(frame, group_cols):
    result = frame.groupby(group_cols, observed=True)['action_taken'].agg(
        applications='size', denials=lambda actions: actions.eq(3).sum()).reset_index()
    result['denial_rate_pct'] = 100 * result['denials'] / result['applications']
    return result

by_band = rates(clean, ['income_band', 'tract_group'])
print('Statewide decided applications with reported income and positive tract percentage:')
print(by_band.round(2).to_string(index=False))
print('\nSensitivity: same calculation by area type:')
print(rates(clean, ['area_type', 'income_band', 'tract_group']).round(2).to_string(index=False))

### Before and after by income band

Here the same lending-decision denominator is used on both sides. The “before” rates include applications with a zero tract percentage; those records were provisionally labelled below median. They do not have a usable tract comparison, so the “after” rates exclude them and applications with missing income. Any difference is a change in which records are counted, not a change in the underlying lender decisions.

In [0]:
before_bands = baseline_table.loc[baseline_table['income_band'].ne('Missing'),
                                  ['income_band', 'tract_group', 'applications', 'denial_rate_pct']]
after_bands = by_band[['income_band', 'tract_group', 'applications', 'denial_rate_pct']].copy()
after_bands['income_band'] = after_bands['income_band'].astype('string')
comparison = before_bands.merge(after_bands, on=['income_band', 'tract_group'],
    suffixes=('_before', '_after'), validate='one_to_one')
print(comparison.round(2).to_string(index=False))
print('\nMissing-income applications in the baseline:',
      int(baseline_table.loc[baseline_table['income_band'].eq('Missing'), 'applications'].sum()))

### Clean-data handoff to exploratory analysis

`clean` is the analysis-ready application table; `by_band` is the statewide summary with counts, denials, and rates. The area-type table splits those same applications into named metropolitan and non-metropolitan groups. The denominator remains decided applications with reported income and a positive tract percentage.

## Exploratory Data Analysis & Visualizations

I compare the income mix of applications across tract groups, then denial rates within income bands and years. Pooling applicants with different income mixes could hide or exaggerate a tract difference.

### Applicant-income distribution by tract group

Each bar shows the income mix of applications in one tract group; its segments sum to 100%. The tract groups contain different numbers of applications, so the bars show shares rather than raw counts.

In [0]:
income_mix = (clean.groupby(['tract_group', 'income_band'], observed=True)
              .size().unstack('income_band').reindex(columns=labels))
income_share = income_mix.div(income_mix.sum(axis=1), axis=0).mul(100)
print('Application counts by tract group and applicant-income band:')
print(income_mix.to_string())
print('\nWithin-tract-group shares (%):')
print(income_share.round(2).to_string())

ax = income_share.loc[['Below median', 'At or above median']].plot(
    kind='barh', stacked=True, figsize=(10, 4),
    color=['#78ADD0', '#4585B1', '#245F8C', '#123B5D'],
    edgecolor='white', linewidth=0.8, width=0.55)
ax.set(xlabel='Share of applications within tract group (%)', ylabel='Tract-income group',
       title='Applicant-income mix differs between the tract groups', xlim=(0, 100))
ax.legend(title='Applicant income', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

Among applications in below-median tracts, 35.08% report income under $50,000, versus 13.30% in tracts at or above the median. At $150,000 or more, the shares are 7.81% and 23.55%, respectively. These figures describe the applications in this file, not all Kansas households.

### Denial rates within applicant-income bands

The table reports applications, denials, and the denial rate for each pairing. The gap is the below-median tract rate minus the at-or-above-median tract rate, measured in percentage points.

In [0]:
rate_pivot = by_band.pivot(index='income_band', columns='tract_group',
                           values='denial_rate_pct').reindex(labels)
rate_gaps = pd.DataFrame({
    'below_median_pct': rate_pivot['Below median'],
    'at_or_above_median_pct': rate_pivot['At or above median'],
    'gap_percentage_points': rate_pivot['Below median'] - rate_pivot['At or above median'],
})
print(rate_gaps.round(2).to_string())

In [0]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=by_band, x='income_band', y='denial_rate_pct',
            hue='tract_group', order=labels, errorbar=None, ax=ax)
ax.set(title='Denial rates are higher in below-median tracts within every income band',
       xlabel='Applicant income band', ylabel='Denied applications (%)')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)
ax.legend(title='Tract income relative to area median')
fig.tight_layout()
plt.show()

The pooled gap is positive in each income band, ranging from 2.07 to 2.95 percentage points. For example, among applicants reporting $50,000–<$100,000, 5,317 of 53,448 applications in below-median tracts were denied (9.95%), versus 5,373 of 75,120 at or above the median (7.15%).

### Patterns across years

Separate panels keep four income bands legible, but all four use the same zero-based y-axis so their rates are comparable. Each point is an annual denial rate for applications in that band and tract group; the two lines compare groups *within the same year*. The smallest annual cell has 812 applications. Year-to-year changes could reflect changes in applicants, lenders, or lending conditions; this extract cannot separate them.

In [0]:
by_year = rates(clean, ['activity_year', 'income_band', 'tract_group'])
g = sns.relplot(data=by_year, x='activity_year', y='denial_rate_pct',
                hue='tract_group', col='income_band', col_wrap=2, kind='line',
                marker='o', facet_kws={'sharey': True}, height=3.3, aspect=1.35)
g.set_axis_labels('Year', 'Denied applications (%)')
g.set_titles('{col_name}')
g.set(ylim=(0, None))
g.fig.suptitle('Annual tract-group denial rates within applicant-income bands', y=1.03)
plt.show()
print('Smallest annual group:', int(by_year['applications'].min()), 'applications')

In [0]:
annual_rates = by_year.pivot_table(
    index=['activity_year', 'income_band'], columns='tract_group',
    values='denial_rate_pct', observed=True)
annual_rates['gap_percentage_points'] = (
    annual_rates['Below median'] - annual_rates['At or above median'])
print('Annual tract-rate gaps by income band (percentage points):')
print(annual_rates['gap_percentage_points'].unstack('income_band').round(2).to_string())
print('\nAnnual cells with a zero or reversed gap:')
print(annual_rates.loc[annual_rates['gap_percentage_points'].le(0),
                       ['Below median', 'At or above median', 'gap_percentage_points']].round(2).to_string())

### Annual gap heatmap

My earlier notebook used heatmaps to scan years and groups. Here each cell shows the below-median tract rate minus the at-or-above-median rate in percentage points, using the same annual calculations as the line chart. Blue cells have a higher below-median rate; red cells have a higher at-or-above-median rate. The two red cells in the under-$50,000 column are the exceptions to the pooled pattern.

In [0]:
gap_grid = annual_rates['gap_percentage_points'].unstack('income_band').reindex(columns=labels)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(gap_grid, annot=True, fmt='.1f', cmap='RdBu', center=0,
            cbar_kws={'label': 'Below minus at-or-above median (percentage points)'}, ax=ax)
ax.set(title='Annual denial-rate gap within each applicant-income band',
       xlabel='Applicant income band', ylabel='Year')
fig.tight_layout()
plt.show()

### Optional statistical tests

Within each applicant-income band, I test whether the denial proportion is the same in below-median and at-or-above-median tracts. The alternative is that the proportions differ. Each two-by-two table counts denied versus not denied among applications with a lending decision. I use a two-sided Pearson chi-square test without continuity correction. All expected cell counts exceed five. I adjust the four p-values with the Holm method and report the observed gaps in percentage points.

These tests assume application records are independent. The extract has no borrower identifier, so I cannot verify that assumption; a person with multiple applications may appear more than once. A small p-value is evidence against equal recorded rates under the test assumptions. It cannot show that tract income caused a denial or account for other applicant and loan characteristics.

In [0]:
from scipy.stats import chi2_contingency

rows = []
for band in labels:
    band_counts = by_band.loc[by_band['income_band'].eq(band)].set_index('tract_group')
    lower = band_counts.loc['Below median']
    higher = band_counts.loc['At or above median']
    contingency = np.array([
        [lower['denials'], lower['applications'] - lower['denials']],
        [higher['denials'], higher['applications'] - higher['denials']],
    ])
    statistic, p_value, _, expected = chi2_contingency(contingency, correction=False)
    assert expected.min() > 5
    rows.append({'income_band': band,
                 'gap_pp': lower['denial_rate_pct'] - higher['denial_rate_pct'],
                 'chi_square': statistic, 'p_value': p_value,
                 'min_expected': expected.min()})

tests = pd.DataFrame(rows)
order = np.argsort(tests['p_value'].to_numpy())
adjusted = np.empty(len(tests))
running = 0.0
for rank, row_index in enumerate(order):
    running = max(running, (len(tests) - rank) * tests.loc[row_index, 'p_value'])
    adjusted[row_index] = min(1.0, running)
tests['holm_p'] = adjusted
print(tests.to_string(index=False, formatters={
    'gap_pp': lambda v: f'{v:.2f}',
    'chi_square': lambda v: f'{v:.2f}',
    'p_value': lambda v: f'{v:.2e}',
    'holm_p': lambda v: f'{v:.2e}',
    'min_expected': lambda v: f'{v:.0f}',
}))

The adjusted p-values are below 0.0001 in all four income bands. The smallest observed gap is 2.07 percentage points, in the under-$50,000 band; the largest is 2.95 points, in the $100,000–<$150,000 band. With this many applications, small differences can produce very small p-values.

## Key Insights / Conclusions

1. **The applicant mix differs substantially between tract groups.** Under-$50,000 applications account for 35.08% of the below-median tract group and 13.30% of the at-or-above-median group. A single statewide tract rate would mix applicants with different reported incomes.
2. **Within every applicant-income band, the pooled denial rate is higher in below-median tracts.** The differences range from 2.07 to 2.95 percentage points. For $50,000–<$100,000, the rate is 9.95% below the tract median versus 7.15% at or above it; the table above gives the application and denial counts.
3. **The same broad pattern holds in named metropolitan areas alone, but annual comparisons are not unanimous.** The named-metro sensitivity table shows a higher pooled rate for below-median tracts in all four bands. Across the 32 year-by-band comparisons, the under-$50,000 band reverses narrowly in 2022 and more visibly in 2025.
4. **Who is excluded affects the reported rate.** After restricting to lending decisions, the rate is 9.52%. Excluding records with nonpositive tract percentages lowers it to 9.10%; excluding missing applicant income raises it to 9.34%. The filters change which applications are counted, not the decisions lenders made.

In this set of recorded Kansas home-purchase applications with decisions, reported income, and a usable tract percentage, below-median tracts have higher denial rates within each broad applicant-income band when years are pooled. That is an association in these application records, not evidence that tract income caused a denial or that otherwise identical applicants would receive different decisions.

## Limitations & Next Steps

The extract omits credit history, debt-to-income ratio, loan terms, property details, lender, and other factors that could explain differences within broad applicant-income bands. Multiple rows may refer to the same borrower, and the seven fields cannot prove whether identical rows are duplicate applications. Missing applicant income and nonpositive tract percentages remove applications from the analysis; the decision log quantifies the exclusions but cannot tell us how the excluded cases would have changed the within-band comparison. Income bands still contain substantial variation within each band.

The `99999` non-metropolitan code is included in the statewide result and separated in the sensitivity table. Confirm the tract-income field's reference population for those records before describing the combined result as a comparison to a uniform metropolitan median. For a stronger follow-up, use narrower income bands, report year-specific group counts, and return to the source files for underwriting and geography fields. Without those checks, keep the conclusion descriptive rather than causal.